# Introdução

# Índice de arquivos

# Objetivos

# TODO


# Importa libs e define funções

In [ ]:
# File management
import zipfile
import gdown
import os
import numpy as np

# Data manipulation
import pandas as pd

# Data splitting
from sklearn.model_selection import train_test_split

# Sampling techniques
from imblearn.under_sampling import ClusterCentroids

# Plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def grab_from_gdrive(file_id, filename):
  parent_dir = '/content'
  url = f"https://drive.google.com/uc?id={file_id}"
  output = os.path.join(parent_dir, filename)
  if not(os.path.isfile(output)):
    gdown.download(url, output=output, quiet=False)

# Carrega alagamentos confirmados

* Transforma em um array de datas

In [ ]:
file_gerados_gpt = 'chamados_gerados_gpt.csv'
file_verificados_gpt = 'chamados_menor3_verificados_gpt.csv'
file_chuvas_gpt = 'sem_chamados_com_chuva_pesada_verificados_gpt.csv'

grab_from_gdrive('11pA4_6NXgG-esXDLrVI9Imwg61kO-_so', file_gerados_gpt)
grab_from_gdrive('1vjV5E5uUm_SRr6-YOr0S8k8aMzl5jERz', file_verificados_gpt)
grab_from_gdrive('1GE5NjwyWynEZpT00gNM43nO5zH-e-N7H', file_chuvas_gpt)

Downloading...
From: https://drive.google.com/uc?id=11pA4_6NXgG-esXDLrVI9Imwg61kO-_so
To: /content/chamados_gerados_gpt.csv
100%|██████████| 13.7k/13.7k [00:00<00:00, 33.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vjV5E5uUm_SRr6-YOr0S8k8aMzl5jERz
To: /content/chamados_menor3_verificados_gpt.csv
100%|██████████| 9.82k/9.82k [00:00<00:00, 26.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GE5NjwyWynEZpT00gNM43nO5zH-e-N7H
To: /content/sem_chamados_com_chuva_pesada_verificados_gpt.csv
100%|██████████| 273/273 [00:00<00:00, 356kB/s]


In [ ]:
df_alagamentos_gerados_gpt = pd.read_csv(file_gerados_gpt, delimiter=",")
df_alagamentos_verificados_gpt = pd.read_csv(file_verificados_gpt, delimiter=",")
df_alagamentos_chuvas_gpt = pd.read_csv(file_chuvas_gpt, delimiter=",")

In [ ]:
df_alagamentos_chuvas_gpt.head()

,Data,Alagamento confirmado?,Evidência,Fonte
0,09/11/2016,Sim,Interrupção da Linha 10-Turquesa da CPTM devid...,Estadão
1,22/12/2016,Sim,Alagamentos em ao menos 4 pontos de Santo Andr...,G1 / Diário do Grande ABC


In [ ]:
datas_alagamentos_confirmados = pd.to_datetime(
    pd.concat(
        [
          df_alagamentos_gerados_gpt['Data'],
          df_alagamentos_verificados_gpt['Data'],
          df_alagamentos_chuvas_gpt['Data']
        ]
    ).unique(), dayfirst=True
)

# Carrega chamados

* Converte a data em datetime
* Filtra por tipo de evento
* Extrai a lista de datas com chamados de enchente


In [ ]:
file_chamados = 'chamados_raw.csv'
grab_from_gdrive('1Fa5TOO-hXAT4l7-IyA51Kswjdxn7WoQD', file_chamados)
df_chamados_raw = pd.read_csv(file_chamados)

Downloading...
From: https://drive.google.com/uc?id=1Fa5TOO-hXAT4l7-IyA51Kswjdxn7WoQD
To: /content/chamados_raw.csv
100%|██████████| 9.08M/9.08M [00:00<00:00, 66.5MB/s]


In [ ]:
df_chamados_dt = df_chamados_raw.copy()
df_chamados_dt['dt'] = pd.to_datetime(df_chamados_dt['DATA'], dayfirst=True)

In [ ]:
# Filtrando por tipo de serviço
enchente = ["809.3 - DDC - Enchente / Inundação / Alagamento - Núcleo",
            "809 - DDC - Enchente / Inundação / Alagamento",
            "809.2 - DDC - Enchente / Inundação / Alagamento - Residência",
            "809.1 - DDC - Enchente / Inundação / Alagamento - Via",
            "809.4 - DDC - Enchente / Inundação / Alagamento - Industria / Comércio"]

df_chamados_temp1 = df_chamados_dt.loc[df_chamados_dt['SERVIÇO'].isin(enchente)].reset_index()
chamados_removal = ["DATA", "ENCAMINHAMENTO", "SOLICITANTE", "SERVIÇO", "index"]
df_chamados_rem = df_chamados_temp1.drop(columns=chamados_removal)

In [ ]:
df_chamados_renamed = df_chamados_rem.rename(columns={"ENDEREÇO": "end"})

In [ ]:
datas_chamados = df_chamados_rem['dt'].unique()

# Verifica chamados

* Verifica quantos chamados já foram confirmados como alagamentos reais
* Verifica quantos chamados não foram confirmados

In [ ]:
datas_chamados_confirmados = np.intersect1d(datas_chamados, datas_alagamentos_confirmados)
datas_chamados_nao_confirmados = np.setdiff1d(datas_chamados, datas_alagamentos_confirmados)

print(f"Dias com chamados confirmados: {datas_chamados_confirmados.shape[0]}")
print(f"Dias com chamados não confirmados: {datas_chamados_nao_confirmados.shape[0]}")

Dias com chamados confirmados: 73
Dias com chamados não confirmados: 507


# Carrega dados CEMADEN

* Carrega dados do google drive com daily metrics

In [ ]:
file_agregado_cemaden = 'df_daily_metrics.csv'
grab_from_gdrive('1wYMSPeli7S2QiZQl0WJR4r3E7u41hQRZ', file_agregado_cemaden)
df_daily_metrics = pd.read_csv(file_agregado_cemaden)

Downloading...
From: https://drive.google.com/uc?id=1wYMSPeli7S2QiZQl0WJR4r3E7u41hQRZ
To: /content/df_daily_metrics.csv
100%|██████████| 6.73M/6.73M [00:00<00:00, 108MB/s]


# Verifica métricas chamados não confirmados

* Filtra dados CEMADEN por dias de chamados não confirmados
* Agrupa por dia, mês e ano e verifica o máximo de cada métrica
* Ordena por máximo de chuva total e verifica os menores índices de chuva total
* Verifica distribuição de max chuva total -> verifica quando deixa de ser 0 (em torno de 5%)
* Define regiões de corte para max chuva total -> de acordo com consulta, a partir de 50 mm de chuva já há um risco razoável de alagamentos; definidos 20, 40, 50 e 100
* Pega a lista de 5 datas para cada recorte de max chuva total -> pega as 5 com menos chuva

Lista 20 -> 2017-03-03 2022-03-28 2017-11-20 2019-03-28 2019-03-21

Lista 40 -> 2022-01-05 2021-12-30 2022-03-16 2020-02-21 2023-02-22

Lista 50 -> 2017-03-06 2019-01-05 2016-06-06 2023-01-20 2020-02-08

Lista 100 -> Tem apenas duas. Nem vale a pena verificar

In [ ]:
df_daily_metrics['dt_txt'] = df_daily_metrics.apply(lambda row: f"{int(row['dia']):02}/{int(row['mes']):02}/{int(row['ano'])}", axis=1)
df_daily_metrics['dt'] = pd.to_datetime(df_daily_metrics['dt_txt'], dayfirst=True)

In [ ]:
df_daily_metrics_nao_confirmados = df_daily_metrics[df_daily_metrics['dt'].isin(datas_chamados_nao_confirmados)]

In [ ]:
df_nao_confirmados_grouped = df_daily_metrics_nao_confirmados.groupby(["ano", "mes", "dia"]).max().reset_index()

In [ ]:
df_nao_confirmados_grouped.sort_values(by="chuva_total").head(5)

,ano,mes,dia,nomeEstacao,latitude,longitude,chuva_total,chuva_max_1h,chuva_max_3h,horas_com_chuva,horas_intensas_10mm,perc_chuva_em_3h,coef_var,dt_txt,dt
7,2017,2,15,Vila Suiça,-23.623,-46.303,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,15/02/2017,2017-02-15
86,2019,5,10,Vila Suiça,-23.636,-46.303,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,10/05/2019,2019-05-10
157,2022,3,23,Vila Vitória,-22.488,-46.303,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,23/03/2022,2022-03-23
58,2019,3,24,Vila Suiça,-23.629,-46.303,0.2,0.2,0.2,1.0,0.0,1.0,4.898979,24/03/2019,2019-03-24
70,2019,4,5,Vila Suiça,-23.629,-46.303,0.2,0.2,0.2,1.0,0.0,1.0,4.898979,05/04/2019,2019-04-05


In [ ]:
df_nao_confirmados_grouped.chuva_total.describe()

,chuva_total
count,189.000000
mean,26.459206
std,25.455761
min,0.000000
25%,4.200000
50%,20.650000
75%,39.700000
max,110.030000


In [ ]:
datas_para_verificar_20 = df_nao_confirmados_grouped.query("chuva_total > 20").sort_values(by="chuva_total").head(5).dt
datas_para_verificar_20 = datas_para_verificar_20.to_list()
for data in datas_para_verificar_20:
  print(data.strftime('%Y-%m-%d'), end=' ')

2017-03-03 2022-03-28 2017-11-20 2019-03-28 2019-03-21 

In [ ]:
datas_para_verificar_40 = df_nao_confirmados_grouped.query("chuva_total > 40").sort_values(by="chuva_total").head(5).dt
datas_para_verificar_40 = datas_para_verificar_40.to_list()
for data in datas_para_verificar_40:
  print(data.strftime('%Y-%m-%d'), end=' ')

2022-01-05 2021-12-30 2022-03-16 2020-02-21 2023-02-22 

In [ ]:
datas_para_verificar_50 = df_nao_confirmados_grouped.query("chuva_total > 50").sort_values(by="chuva_total").head(5).dt
datas_para_verificar_50 = datas_para_verificar_50.to_list()
for data in datas_para_verificar_50:
  print(data.strftime('%Y-%m-%d'), end=' ')

2017-03-06 2019-01-05 2016-06-06 2023-01-20 2020-02-08 

In [ ]:
datas_para_verificar_100 = df_nao_confirmados_grouped.query("chuva_total > 100").sort_values(by="chuva_total").head(5).dt
datas_para_verificar_100 = datas_para_verificar_100.to_list()
for data in datas_para_verificar_100:
  print(data.strftime('%Y-%m-%d'), end=' ')

2020-03-02 2022-11-25 

In [ ]:
df_nao_confirmados_grouped.query("chuva_total > 60").shape

(20, 15)

# Verifica datas de cada recorte

* Verificados com GPT search

Lista 20: Após uma pesquisa detalhada, não foram encontradas notícias de enchentes ou alagamentos em Santo André nas datas específicas fornecidas. As ocorrências mais próximas identificadas são:


    07/03/2017: Forte temporal causou alagamentos em Santo André e região. (dgabc.com.br )

    11/03/2019: Temporal resultou em alagamentos significativos em Santo André. (dgabc.com.br )

    04/01/2022: Chuva intensa provocou alagamentos em Santo André. (dgabc.com.br )
     
Como essas datas não coincidem exatamente com as fornecidas, não há registros de enchentes ou alagamentos em Santo André nos dias específicos mencionados.

Lista 40: Após verificar as datas fornecidas, não encontrei notícias de enchentes ou alagamentos em Santo André exatamente nos dias especificados. No entanto, houve registros de alagamentos em datas próximas. Abaixo, apresento uma tabela com as datas solicitadas, indicando se houve enchente e fornecendo links para as fontes correspondentes:

| Data       | Houve enchente? | Fonte                                                                 |
|------------|-----------------|-----------------------------------------------------------------------|
| 2022-01-05 | Não             | [Chuva de uma hora no 1º dia útil de 2022 mais uma vez castiga o Grande ABC - 03/01/2022](https://www.dgabc.com.br/Noticia/3824045/chuva-de-uma-hora-no-1-dia-util-de-2022-mais-uma-vez-castiga-o-grande-abc) |
| 2021-12-30 | Não             | [Chuva de uma hora no 1º dia útil de 2022 mais uma vez castiga o Grande ABC - 03/01/2022](https://www.dgabc.com.br/Noticia/3824045/chuva-de-uma-hora-no-1-dia-util-de-2022-mais-uma-vez-castiga-o-grande-abc) |
| 2022-03-16 | Não             | [Chuva de uma hora no 1º dia útil de 2022 mais uma vez castiga o Grande ABC - 03/01/2022](https://www.dgabc.com.br/Noticia/3824045/chuva-de-uma-hora-no-1-dia-util-de-2022-mais-uma-vez-castiga-o-grande-abc) |
| 2020-02-21 | Não             | [Chuva de uma hora no 1º dia útil de 2022 mais uma vez castiga o Grande ABC - 03/01/2022](https://www.dgabc.com.br/Noticia/3824045/chuva-de-uma-hora-no-1-dia-util-de-2022-mais-uma-vez-castiga-o-grande-abc) |
| 2023-02-22 | Não             | [Chuva causa alagamentos no Grande ABC nesta quinta (23) - 23/02/2023](https://www.dgabc.com.br/Noticia/3950980/chuva-causa-alagamentos-no-grande-abc-nesta-quinta-23) |

Embora não haja registros de enchentes exatamente nas datas mencionadas, é importante notar que ocorreram alagamentos em Santo André em datas próximas, como em 3 de janeiro de 2022 e 23 de fevereiro de 2023.

Lista 50: Após verificar as datas fornecidas, encontrei registros de alagamentos em Santo André nos dias 6 de março de 2017 e 19 de janeiro de 2019. Para as demais datas, não foram encontradas notícias específicas sobre enchentes ou alagamentos no município.

| Data       | Houve Enchente? | Fonte                                                                                   |
|------------|-----------------|-----------------------------------------------------------------------------------------|
| 2017-03-06 | Sim             | [Forte temporal alaga municípios da região - 07/03/2017](https://www.dgabc.com.br/Noticia/2513871/forte-temporal-alaga-municipios-da-regiao) |
| 2019-01-05 | Sim             | [Chuva forte provoca pontos de alagamento em cinco cidades - 19/01/2019](https://www.dgabc.com.br/Noticia/3007327/chuva-forte-provoca-pontos-de-alagamento-em-cinco-cidades) |
| 2016-06-06 | Não encontrado  |                                                                                         |
| 2023-01-20 | Não encontrado  |                                                                                         |
| 2020-02-08 | Não encontrado  |                                                                                         |

Observação: As notícias encontradas referem-se a datas próximas às solicitadas, especificamente um dia após (7 de março de 2017) e 14 dias após (19 de janeiro de 2019) as datas fornecidas. Recomendo verificar manualmente as fontes para confirmar as informações.

# Testa com todas as datas acima de 60

Os primeiros recortes não funcionaram, mas o recorte para precipitação acima de 60 tem apenas 20 valores. Vamos colocá-los direto no deep research do GPT:
Perfeito, vou buscar informações sobre possíveis enchentes ou alagamentos em Santo André nas 20 novas datas que você indicou, mantendo os mesmos critérios de exatidão e confiabilidade das fontes. Te aviso assim que a nova tabela estiver pronta!

| Data       | Houve Enchente? | Fonte (Notícia)                                                 |
|------------|-----------------|------------------------------------------------------------------|
| 2016-01-09 | Não             | Notícia de um dia depois: *Chuva forte causa primeira enchente do ano na região* ([Chuva forte causa primeira enchente do ano na região - 10/01/2016 | Diário do Grande ABC](https://www.dgabc.com.br/noticia/1690202/chuva-forte-causa-primeira-enchente-do-ano-na-regiao#:~:text=A%20chuva%20forte%20que%20atingiu,Trens%20Metropolitanos%20de%20S%C3%A3o%20Paulo)) (publicada em 10/01/2016) – relata enchentes em Santo André no dia 09/01. |
| 2017-01-17 | Sim             | *Região mantém normalidade após intensas chuvas* – Diário do Grande ABC (17/01/2017) ([Região mantém normalidade após intensas chuvas - 17/01/2017 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2503014/regiao-mantem-normalidade-apos-intensas-chuvas#:~:text=Em%20Santo%20Andr%C3%A9%20foram%20registrados,est%C3%A3o%20com%20seus%20n%C3%ADveis%20normais)) (alagamentos registrados em Santo André). |
| 2018-01-23 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2018-03-20 | Sim             | *Forte chuva causa transtornos no Grande ABC* – Diário do Grande ABC (20/03/2018) ([Forte chuva causa transtornos no Grande ABC | Diário do Grande ABC - Notícias e informações](http://www.dgabc.com.br/Mobile/Noticia/2871267/forte-chuva-causa-transtornos-no-grande-abc#:~:text=Em%20Santo%20Andr%C3%A9%2C%20a%20Defesa,os%20pluvi%C3%B4metros%20marcaram%2056%20mil%C3%ADmetros)) (diversos pontos de alagamento em Santo André). |
| 2018-04-15 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2019-02-04 | Sim             | *Após chuvas, asfalto cede e abre crateras…* – Diário do Grande ABC (04/02/2019) ([Após chuvas, asfalto cede e abre crateras ao longo da Avenida Guido Aliberti - 04/02/2019 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/3012776/chuva-provoca-pontos-de-alagamento-e-transito-na-regiao#:~:text=Ap%C3%B3s%20a%20chuva%20que%20atingiu,com%20a%20Rua%20Serafim%20Carlos)) ([Após chuvas, asfalto cede e abre crateras ao longo da Avenida Guido Aliberti - 04/02/2019 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/3012776/chuva-provoca-pontos-de-alagamento-e-transito-na-regiao#:~:text=Uma%20%C3%A1rvore%20de%20grande%20porte,alguns%20sem%C3%A1foros%20do%20Centro%20tiveram)) (pontos de alagamento registrados em Santo André). |
| 2019-02-16 | Sim             | *Chuva e vento trazem destruição* – Diário do Grande ABC (16/02/2019) ([Chuva e vento trazem destruição - 16/02/2019 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/3016343/chuva-e-vento-trazem-destruicao#:~:text=De%20acordo%20com%20a%20Defesa,Estados%3B%20e%20Avenida%20Santos%20Dumont)) (várias vias de Santo André ficaram alagadas). |
| 2019-04-08 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2020-03-02 | Não             | Notícia de um dia depois: na chuva de 03/03/2020, *a Avenida dos Estados ficou alagada* em Santo André ([Chuva causa desabamento de casa em Mauá, alaga estação em Santo André e se espalha por SP](https://noticias.uol.com.br/ultimas-noticias/agencia-estado/2025/03/31/chuva-causa-desabamento-de-casa-em-maua-alaga-estacao-em-santo-andre-e-se-espalha-por-sp.htm#:~:text=Em%20Santo%20Andr%C3%A9%2C%20a%20chuva,de%20alagamento%20na%20avenida%20Kennedy)) (publicada em 03/03/2020). |
| 2020-03-03 | Sim             | *Chuva alaga ruas e paralisa trens no ABC* – (03/03/2020) ([Chuva causa desabamento de casa em Mauá, alaga estação em Santo André e se espalha por SP](https://noticias.uol.com.br/ultimas-noticias/agencia-estado/2025/03/31/chuva-causa-desabamento-de-casa-em-maua-alaga-estacao-em-santo-andre-e-se-espalha-por-sp.htm#:~:text=Em%20Santo%20Andr%C3%A9%2C%20a%20chuva,de%20alagamento%20na%20avenida%20Kennedy)) (Avenida dos Estados intransitável em Santo André). |
| 2020-06-27 | Não             | *Chuvas provocam queda de muro de cemitério* – ABCD Jornal (27/06/2020) ([Chuvas provocam queda de muro de cemitério em Santo André](https://abcdjornal.com.br/chuvas-provocam-queda-de-muro-de-cemiterio-em-santo-andre/#:~:text=Parte%20do%20muro%20cemit%C3%A9rio%20da,devido%20a%20forte%20chuva)) ([Chuvas provocam queda de muro de cemitério em Santo André](https://abcdjornal.com.br/chuvas-provocam-queda-de-muro-de-cemiterio-em-santo-andre/#:~:text=N%C3%A3o%20houve%20nenhuma%20outra%20ocorr%C3%AAncia,a%20chuva%20em%20Santo%20Andr%C3%A9)) (chuva forte, **sem alagamentos** registrados em Santo André). |
| 2021-02-02 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2021-02-12 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2021-12-14 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2022-01-07 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2022-01-18 | Sim             | *Temporal castiga Santo André* – Jornal da Record (18/01/2022) ([Temporal castiga Santo André (SP) na tarde desta terça-feira (18) – Record](https://record.r7.com/jornal-da-record/videos/temporal-castiga-santo-andre-sp-na-tarde-desta-terca-feira-18-24052022/#:~:text=Os%20rios%20que%20cortam%20a,resgatado%20no%20meio%20do%20alagamento)) (rios transbordaram e o centro da cidade ficou alagado). |
| 2022-09-27 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2022-11-25 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |
| 2023-01-31 | Sim             | *Chuva provoca alagamentos em vários pontos do ABC* – Repórter Diário (31/01/2023) ([Chuva provoca pontos de alagamentos em vários pontos do ABC](https://www.reporterdiario.com.br/noticia/3215792/chuva-provoca-pontos-de-alagamentos-em-varios-pontos-do-abc/#:~:text=J%C3%A1%20em%20Santo%20Andr%C3%A9%2C%20endere%C3%A7os,e%20Centreville%20tamb%C3%A9m%20est%C3%A3o%20alagados)) (bairros de Santo André com ruas alagadas). |
| 2023-02-02 | Não             | Nenhum registro encontrado de enchente/alagamento nessa data.    |

In [ ]:
df_nao_confirmados_grouped.query("chuva_total > 60").shape

(20, 15)

In [ ]:
df_nao_confirmados_grouped.query("chuva_total > 60").dt.to_list()
for data in df_nao_confirmados_grouped.query("chuva_total > 60").dt.to_list():
  print(data.strftime('%Y-%m-%d'), end=' ')

2016-01-09 2017-01-17 2018-01-23 2018-03-20 2018-04-15 2019-02-04 2019-02-16 2019-04-08 2020-03-02 2020-03-03 2020-06-27 2021-02-02 2021-02-12 2021-12-14 2022-01-07 2022-01-18 2022-09-27 2022-11-25 2023-01-31 2023-02-02 

In [ ]:
from datetime import datetime
confirmados_mais_60 = np.array([
    datetime(2017, 1, 17),
    datetime(2018, 3, 20),
    datetime(2019, 2, 4),
    datetime(2019, 2, 16),
    datetime(2020, 3, 3),
    datetime(2022, 1, 18),
    datetime(2023, 1, 31)
])

In [ ]:
confirmados_mais_60 = pd.DatetimeIndex(confirmados_mais_60)
confirmados_mais_60

DatetimeIndex(['2017-01-17', '2018-03-20', '2019-02-04', '2019-02-16',
               '2020-03-03', '2022-01-18', '2023-01-31'],
              dtype='datetime64[ns]', freq=None)

# Testa entre 30 e 60
Perfeito! Vou agora verificar se houve registros de enchentes ou alagamentos em Santo André nas novas datas fornecidas, seguindo os mesmos critérios. Assim que a tabela estiver pronta, te aviso!

## Registros de Enchentes/Alagamentos em Santo André, SP

| Data       | Houve Enchente? | Fonte (Notícia)                                                            |
|------------|-----------------|---------------------------------------------------------------------------|
| 2016-06-06 | **Não**         | Notícia de um dia depois: *Chuva atípica causa problemas* (Diário do Grande ABC) ([Chuva atípica causa problemas - 07/06/2016 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/1973621/chuva-atipica-causa-problemas#:~:text=contabilizou%20pontos%20de%20alagamento%2C%20quedas,muros%2C%20al%C3%A9m%20de%20tr%C3%A2nsito%20complicado)) |
| 2016-10-13 | **Não**         | Notícia de um dia depois: *Chuva forte deixa região em estado de alerta* (Diário do Grande ABC) ([Chuva forte deixa região em estado de alerta | Diário do Grande ABC - Notícias e informações](https://www.dgabc.com.br/Mobile/Noticia/2392684/chuva-forte-deixa-regiao-em-estado-de-alerta#:~:text=Forte%20chuva%20e%20rajadas%20de,n%C3%A3o%20trouxe%20transtornos%20%C3%A0%20popula%C3%A7%C3%A3o)) ([Chuva forte deixa região em estado de alerta | Diário do Grande ABC - Notícias e informações](https://www.dgabc.com.br/Mobile/Noticia/2392684/chuva-forte-deixa-regiao-em-estado-de-alerta#:~:text=Segundo%20o%20Corpo%20de%20Bombeiros%2C,pontos%20de%20alagamento%20nem%20deslizamentos)) |
| 2016-11-03 | **Não**         | Notícia de um dia depois: *Chuva deixa pontos de alagamento na região* (Diário do Grande ABC) ([Chuva deixa pontos de alagamento na região  | Diário do Grande ABC - Notícias e informações](https://www.dgabc.com.br/Mobile/Noticia/2447399/chuva-deixa-pontos-de-alagamento#:~:text=Em%20Santo%20Andr%C3%A9%2C%20houve%20alagamentos,Gales%20e%20Rua%20Gago%20Coutinho)) |
| 2016-12-21 | **Sim**        | *Chuva causa transtornos à população* (Diário do Grande ABC) ([Chuva causa transtornos à população - 21/12/2016 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2498009/chuva-causa-transtornos-a-populacao#:~:text=Uma%20hora%20e%20meia,sem%C3%A1foros%20desligados%20complementaram%20os%20transtornos)) |
| 2017-03-02 | **Não**         | Notícia de um dia depois: *Forte chuva causa estragos no Grande ABC* (Diário do Grande ABC) ([Forte chuva causa estragos no Grande ABC - 03/03/2017 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2513152/forte-chuva-causa-estragos-no-grande-abc#:~:text=Em%20Santo%20Andr%C3%A9%2C%20o%20Semasa,Branco%2C%20no%20bairro%20Sacadura%20Cabral)) |
| 2017-03-06 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2017-03-08 | **Sim**        | *Forte temporal alaga municípios da região* (Diário do Grande ABC) ([](http://www.santoandre.sp.gov.br/pesquisa/arquivoper.asp?file=170308003.pdf#:~:text=Em%20Santo%20Andr%C3%A9%2C%20a%20chuva,Marta%20de%20Oliveira%20Lino%2C%2059)) |
| 2017-11-19 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2018-11-30 | **Sim**        | *Chuva forte provoca queda de telhado e alagamentos* (Diário do Grande ABC) ([Chuva forte em Santo André provoca queda de telhado de supermercado - 30/11/2018 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2993605/tempestade-com-rajadas-de-vento-atinge-regiao#:~:text=A%20situa%C3%A7%C3%A3o%20se%20normalizou%20aos,na%20linha%20Turquesa%20da%20CPTM)) |
| 2019-01-05 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-01-19 | **Sim**        | *Chuva forte provoca pontos de alagamento* (Diário do Grande ABC) ([Chuva forte provoca pontos de alagamento em cinco cidades - 19/01/2019 | Diário do Grande ABC](https://www.dgabc.com.br/noticia/3007327/chuva-forte-provoca-pontos-de-alagamento-em-cinco-cidades#:~:text=Na%20esta%C3%A7%C3%A3o%20da%20CPTM%20,restabelecida%20por%20volta%20das%2018h)) |
| 2019-02-12 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-02-17 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-02-20 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-03-12 | **Não**         | Notícia de um dia antes: *Após chuvas, São Bernardo decreta calamidade; Santo André pede doações* (UOL) ([Após chuvas, São Bernardo decreta calamidade; Santo André pede doações - 11/03/2019 - UOL Notícias](https://noticias.uol.com.br/cotidiano/ultimas-noticias/2019/03/11/apos-chuvas-sao-bernardo-decreta-calamidade-santo-andre-pede-doacoes.htm#:~:text=Houve%2012%20mortes%20em%20decorr%C3%AAncia,que%20estava%20na%20garupa%20sobreviveu)) |
| 2019-03-14 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-03-17 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-04-06 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2019-07-05 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2020-02-08 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2020-02-09 | **Não**         | Notícia de um dia depois: *Temporal castiga… 20 bairros do ABC* (Repórter Diário) ([Temporal castiga e atinge pelo menos 20 bairros do ABC](https://www.reporterdiario.com.br/noticia/2784845/temporal-castiga-e-atinge-pelo-menos-20-bairros-do-abc/#:~:text=J%C3%A1%20em%20Santo%20Andr%C3%A9%20107,Alzira%20e%20Jardim%20Santo%20Ant%C3%B4nio)) |
| 2020-02-11 | **Não**         | Notícia de um dia antes: *Temporal castiga… 20 bairros do ABC* (Repórter Diário) ([Temporal castiga e atinge pelo menos 20 bairros do ABC](https://www.reporterdiario.com.br/noticia/2784845/temporal-castiga-e-atinge-pelo-menos-20-bairros-do-abc/#:~:text=J%C3%A1%20em%20Santo%20Andr%C3%A9%20107,Alzira%20e%20Jardim%20Santo%20Ant%C3%B4nio)) |
| 2020-02-19 | **Não**         | Notícia de um dia antes: *Chuva causa transtornos...* (Diário do Grande ABC) ([Chuva causa transtornos em Santo André e Mauá - 18/02/2020 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/3328306/chuva-causa-transtornos-no-grande-abc#:~:text=Em%20Santo%20Andr%C3%A9%20o%20rio,N%C3%A3o%20houve%20registro%20de%20feridos)) |
| 2020-02-21 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2020-02-24 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2020-02-26 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2020-11-26 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-01-01 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-02-18 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-02-27 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-04-18 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-12-13 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-12-30 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2021-12-31 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-01-05 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-01-06 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-01-10 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-01-14 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-01-19 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-02-17 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-03-10 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-03-11 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-03-13 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-03-15 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2022-03-16 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2023-01-20 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2023-01-30 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2023-02-13 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2023-02-22 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2023-03-08 | **Não**         | Notícia de um dia depois: *Chuva forte causa alagamentos...* (DGABC) ([Chuva forte causa alagamentos em Santo André nesta quinta (9) - 09/03/2023 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/3953221/chuva-forte-causa-alagamentos-em-santo-andre-nesta-quinta-9#:~:text=A%20forte%20chuva%20da%20tarde,S%C3%A3o%20Caetano%20e%20Santo%20Andr%C3%A9)) |
| 2023-03-10 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |
| 2023-03-12 | **Não**         | *(Nenhuma ocorrência registrada em notícia)*                               |

In [ ]:
df_nao_confirmados_grouped.query("chuva_total > 30 and chuva_total <= 60").dt.to_list()
for data in df_nao_confirmados_grouped.query("chuva_total > 30 and chuva_total <= 60").dt.to_list():
  print(data.strftime('%Y-%m-%d'), end=' ')

2016-06-06 2016-10-13 2016-11-03 2016-12-21 2017-03-02 2017-03-06 2017-03-08 2017-11-19 2018-11-30 2019-01-05 2019-01-19 2019-02-12 2019-02-17 2019-02-20 2019-03-12 2019-03-14 2019-03-17 2019-04-06 2019-07-05 2020-02-08 2020-02-09 2020-02-11 2020-02-19 2020-02-21 2020-02-24 2020-02-26 2020-11-26 2021-01-01 2021-02-18 2021-02-27 2021-04-18 2021-12-13 2021-12-30 2021-12-31 2022-01-05 2022-01-06 2022-01-10 2022-01-14 2022-01-19 2022-02-17 2022-03-10 2022-03-11 2022-03-13 2022-03-15 2022-03-16 2023-01-20 2023-01-30 2023-02-13 2023-02-22 2023-03-08 2023-03-10 2023-03-12 

In [ ]:
confirmados_30_60 = pd.to_datetime([
    "2016-12-21",
    "2017-03-08",
    "2018-11-30",
    "2019-01-19"
])

confirmados_30_60 = pd.DatetimeIndex(confirmados_30_60)

# Agrega alagamentos confirmados

* Adiciona confirmados finais do GPT às datas confirmadas
* Verifica quantidade de alagamentos por ano
* Verifica quantos eventos subsequentes temos

In [ ]:
total_alagamentos = np.unique(np.concatenate((datas_alagamentos_confirmados, confirmados_mais_60, confirmados_30_60)))

In [ ]:
df_alagamentos = pd.DataFrame(total_alagamentos, columns=['dt'])
df_alagamentos['dt'] = pd.to_datetime(df_alagamentos['dt'])
df_alagamentos['ano'] = df_alagamentos['dt'].dt.year
df_alagamentos['mes'] = df_alagamentos['dt'].dt.month
df_alagamentos['dia'] = df_alagamentos['dt'].dt.day

In [ ]:
df_alagamentos.groupby("ano").dia.count()

,dia
ano,
2016,17
2017,12
2018,23
2019,13
2020,7
2021,5
2022,14
2023,8
2024,4


In [ ]:
df_alagamentos['date_diff'] = df_alagamentos['dt'].diff().dt.days

Quantidade de dias que foram logo em seguida de um evento anterior

In [ ]:
df_alagamentos.query("date_diff == 1").reset_index().groupby("ano").dia.count()

,dia
ano,
2016,6
2017,4
2018,8
2019,4
2022,3
2023,1


In [ ]:
for date in df_alagamentos.query("date_diff == 1").dt.to_list():
  print(date.strftime('%Y-%m-%d'), end=' ')

2016-01-11 2016-01-16 2016-02-16 2016-02-25 2016-12-22 2016-12-28 2017-01-17 2017-03-08 2017-04-07 2017-12-21 2018-03-15 2018-03-18 2018-03-21 2018-03-24 2018-03-27 2018-03-30 2018-11-24 2018-11-25 2019-01-11 2019-01-19 2019-02-22 2019-03-11 2022-01-04 2022-03-07 2022-03-08 2023-01-16 

In [ ]:
df_alagamentos_sem_subsequentes = df_alagamentos.query("date_diff != 1")

In [ ]:
df_alagamentos_sem_subsequentes.groupby("ano").dia.count()

,dia
ano,
2016,11
2017,8
2018,15
2019,9
2020,7
2021,5
2022,11
2023,7
2024,4


# Verifica incidentes subsequentes

* Usa o GPT para verificar entre as datas subsequentes, quais realmente tiveram incidentes subsequentes e quais são apenas notícias sobre o mesmo incidente.
* Agrega o resultado final dos alagamentos que podemos utilizar com confiança

Resultado pesquisa GPT:
Perfeito, vou agora verificar cuidadosamente cada grupo de datas consecutivas para determinar se houve mais de um evento distinto de alagamento ou apenas um, e retornarei a análise em tabela conforme solicitado.

| **Grupo de Datas**                   | **Nº de Eventos Confirmados** | **Datas com Incidente**              | **Explicação e Referência**                                                                                                                                                                   |
|--------------------------------------|------------------------------|--------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **2016-01-10 e 2016-01-11**          | 1                            | 09/01/2016                           | Houve um único temporal, ocorrido em 9 de janeiro de 2016. A notícia de 10/01 descreve a “primeira enchente do ano” provocada pela chuva forte **na tarde de ontem** (dia 9) ([Chuva forte causa primeira enchente do ano na região - 10/01/2016 | Diário do Grande ABC](https://www.dgabc.com.br/noticia/1690202/chuva-forte-causa-primeira-enchente-do-ano-na-regiao#:~:text=A%20chuva%20forte%20que%20atingiu,Trens%20Metropolitanos%20de%20S%C3%A3o%20Paulo)). Não há registro de novo alagamento em 11/01; as reportagens neste dia apenas repercutem o evento anterior. |
| **2016-01-15 e 2016-01-16**          | 1                            | 15/01/2016                           | Apenas um evento de alagamento foi confirmado. Uma forte chuva atingiu Santo André em 15 de janeiro, causando pontos de enchente reportados no dia seguinte. Não houve outro temporal significativo em 16/01, indicando tratar-se do mesmo incidente (15/01) sendo noticiado no dia posterior. |
| **2016-02-15 e 2016-02-16**          | 1                            | 15/02/2016                           | Um único temporal de grande intensidade ocorreu em 15 de fevereiro de 2016. A matéria de 16/02 relata que **vias do Grande ABC ficaram debaixo d’água ontem** (dia 15) devido a três horas de chuva intensa ([Enchentes tomam as ruas e uma pessoa desaparece - 16/02/2016 | Diário do Grande ABC](https://www.dgabc.com.br/noticia/1787484/enchentes-tomam-as-ruas-e-uma-pessoa-desaparece#:~:text=Tr%C3%AAs%20horas%20de%20chuva%20intensa,Guido%20Aliberti%2C%20em%20S%C3%A3o%20Caetano)). Ou seja, 16/02 não teve novo alagamento, apenas a continuação da cobertura do evento do dia 15. |
| **2016-02-24 e 2016-02-25**          | 1                            | 24/02/2016                           | Um único evento de enchente ocorreu em 24 de fevereiro de 2016. No dia 25, tanto a Prefeitura quanto a imprensa relataram os estragos da chuva **de ontem (24/2)**, incluindo transbordamento do córrego Oratório e dezenas de casas inundadas ([ » Chuvas de hoje em Santo André](https://portais.santoandre.sp.gov.br/semasa/2016/02/24/chuvas-de-hoje-em-santo-andre/#:~:text=Santo%20Andr%C3%A9%2C%2024%20de%20fevereiro,S%C3%A1%20e%20Parque%20Novo%20Orat%C3%B3rio)). Não houve novo alagamento em 25/02 – tratava-se do mesmo incidente do dia 24. |
| **2016-12-21 e 2016-12-22**          | 1                            | 21/12/2016                           | Confirmou-se apenas um temporal, no dia 21 de dezembro de 2016. O primeiro dia do verão (21/12) teve **forte chuva e pontos de alagamento** ([Verão começa com temporal; Natal terá sol - 22/12/2016 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2498212/verao-comeca-com-temporal-natal-tera-sol#:~:text=O%20primeiro%20dia%20do%20ver%C3%A3o,de%20vento%20e%20descargas%20el%C3%A9tricas)) em Santo André. As notícias de 22/12 referem-se a essa enchente do dia anterior, sem registro de novo episódio no dia 22. |
| **2016-12-27 e 2016-12-28**          | 1                            | 27/12/2016                           | Somente um evento de alagamento ocorreu em 27 de dezembro de 2016. Reportagem de 29/12 relembra que na **tarde de terça-feira** (27/12) um temporal causou enchentes em Santo André ([Após chuva, hora é de calcular prejuízo  - 29/12/2016 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2499196/apos-chuva-hora-e-de-calcular-prejuizo#:~:text=Moradores%20de%20Mau%C3%A1%20e%20Santo,e%20sujeira%20nas%20%C3%A1reas%20alagadas)), e o dia 28 foi dedicado à limpeza e contabilização de prejuízos, não a um novo alagamento. |
| **2017-01-16 e 2017-01-17**          | 1                            | 16/01/2017                           | Apenas um incidente de enchente, ligado à chuva forte entre a noite de 16/01 e a madrugada de 17/01. Dados meteorológicos mostram que em Santo André choveu 54 mm entre 22h de segunda (16) e 2h da manhã de terça (17), caracterizando um só evento contínuo ([Chuva voltou com força para a Grande SP - Climatempo](https://www.climatempo.com.br/noticia/2017/01/16/chuva-deixa-sp-em-alerta-6627#:~:text=Chuva%20voltou%20com%20for%C3%A7a%20para,per%C3%ADodo%20e%20em%20Guarulhos)). Não há registro de outro alagamento separado no dia 17. |
| **2017-03-07 e 2017-03-08**          | 1                            | 07/03/2017                           | Um único temporal em 7 de março de 2017 causou inundações. A noite de 07/03 registrou transbordamento de córregos e **grandes alagamentos** em municípios do Grande ABC ([Balanço do temporal na Grande SP 07/03/17 | Climatempo](https://www.climatempo.com.br/noticia/2017/03/07/temporal-na-grande-sp-9531#:~:text=Rios%20voltam%20a%20transbordar)), incluindo Santo André. No dia 08/03 não houve nova chuva forte – as notícias apenas repercutiram o caos do dia 7. |
| **2017-04-06 e 2017-04-07**          | 1                            | 07/04/2017                           | Houve somente um evento, no dia 7 de abril de 2017. Uma tempestade nessa data provocou danos como o colapso de uma ponte sobre o rio Tamanduateí em Santo André ([Semasa inicia obra de reconstrução de...](https://web.santoandre.sp.gov.br/portal/noticias/0/3/11251/semasa-inicia-obra-de-reconstrucao-de-ponte-sobre-o-rio-tamanduatei#:~:text=Estrutura%20entrou%20em%20colapso%20em,planejada%20para%20suportar%20fortes%20chuvas)). Não ocorreu outro temporal em 06/04; a enchente relevante foi a do dia 07/04. |
| **2017-12-20 e 2017-12-21**          | 1                            | 20/12/2017                           | Apenas um evento de chuva intensa ocorreu em 20 de dezembro de 2017. Santo André sofreu alagamentos nesse dia (avenidas dos Estados, Industrial, etc.) devido à forte chuva da tarde ([Chuva castiga Grande ABC na tarde desta quarta-feira - 20/12/2017 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/2812823/chuva-castiga-grande-abc-na-tarde-desta-quarta-feira#:~:text=Santo%20Andr%C3%A9%20sofre%20com%20a,tentativa%20de%20cruzar%20a%20cheia)). No dia 21/12 não houve novo alagamento; as informações divulgadas referiam-se aos transtornos causados pela chuva do dia 20. |
| **2018-03-14 e 2018-03-15**          | 1                            | 14/03/2018                           | Um único episódio de enchente aconteceu em 14 de março de 2018. As notícias de 15/03 tratam do temporal do dia anterior (14/03) e seus efeitos, não havendo registro de outra enchente no dia 15. Portanto, foi um só evento de alagamento, ocorrido em 14 de março. |
| **2018-03-17 e 2018-03-18**          | 1                            | 17/03/2018                           | Somente um evento confirmado, em 17 de março de 2018. Houve forte chuva e alagamentos em Santo André no dia 17; as menções no dia 18/03 dizem respeito a esse mesmo episódio (por exemplo, balanços de ocorrências e limpeza), não indicando novo alagamento no domingo. |
| **2018-03-20 e 2018-03-21**          | 1                            | 20/03/2018                           | Apenas um temporal, ocorrido em 20 de março de 2018. A cobertura jornalística de 21/03 faz referência às enchentes do dia 20, sem apontar chuva severa adicional no dia 21. Assim, trata-se de um único evento (20/03) reportado em dias consecutivos. |
| **2018-03-23 e 2018-03-24**          | 1                            | 23/03/2018                           | Um único evento, no dia 23 de março de 2018. A notícia de 24/03 relata justamente a enchente **ocorrida no dia anterior**, 23/03 (conforme o exemplo dado), deixando claro que não houve novo alagamento em 24/03, mas sim a repercussão do caso do dia 23. |
| **2018-03-26 e 2018-03-27**          | 1                            | 26/03/2018                           | Houve um só episódio de alagamento em 26 de março de 2018. As informações divulgadas em 27/03 correspondem às consequências da forte chuva de 26/03 – não se identifica outro temporal significativo em 27 de março. |
| **2018-03-29 e 2018-03-30**          | 1                            | 29/03/2018                           | Somente um evento confirmado, em 29 de março de 2018. A data seguinte (30/03) não apresentou novo alagamento em Santo André; as reportagens deste dia referem-se ainda à tempestade e alagamentos registrados em 29/03. |
| **2018-11-23, 2018-11-24 e 2018-11-25** | 1                         | 23/11/2018                           | Um único evento de grandes proporções ocorreu em 23 de novembro de 2018. Nesse temporal, Santo André foi uma das cidades mais afetadas, com ruas inundadas e pessoas ilhadas ([Chuva alaga vias na Grande São Paulo; duas pessoas morreram - 23/11/2018 - UOL Notícias](https://noticias.uol.com.br/cotidiano/ultimas-noticias/2018/11/23/chuva-alaga-vias-na-grande-sao-paulo-e-provoca-desabamento-de-casa-em-maua.htm#:~:text=A%20forte%20chuva%20que%20atingiu,em%20S%C3%A3o%20Bernardo%20do%20Campo)) ([Chuva alaga vias na Grande São Paulo; duas pessoas morreram - 23/11/2018 - UOL Notícias](https://noticias.uol.com.br/cotidiano/ultimas-noticias/2018/11/23/chuva-alaga-vias-na-grande-sao-paulo-e-provoca-desabamento-de-casa-em-maua.htm#:~:text=match%20at%20L615%20Em%20Santo,das%20Na%C3%A7%C3%B5es%2C%20entre%20outros%20locais)). No dia 24/11 houve atualização de informações (inclusive confirmação de mortes em cidades vizinhas), mas sem nova chuva forte. O dia 25/11 seguiu sem novos alagamentos relevantes, apenas com desdobramentos do desastre de 23/11. |
| **2019-01-10 e 2019-01-11**          | 1                            | 10/01/2019                           | Apenas um evento de enchente confirmado, em 10 de janeiro de 2019. Uma forte chuva atingiu o ABC na noite do dia 10, causando alagamentos em Santo André. As notícias de 11/01 relatam esses transtornos (pontos alagados, interrupções), não havendo registro de outro temporal no dia 11. |
| **2019-01-18 e 2019-01-19**          | 1                            | 18/01/2019                           | Somente um evento, em 18 de janeiro de 2019. A chuva intensa desta data provocou alagamentos que foram noticiados até o dia 19/01. Não há indicação de que tenha ocorrido nova enchente separada no dia 19; tratou-se do mesmo episódio iniciado em 18/01. |
| **2019-02-21 e 2019-02-22**          | 1                            | 21/02/2019                           | Um único evento registrado, no dia 21 de fevereiro de 2019. As matérias de 22/02 referem-se aos efeitos da forte chuva de 21/02 (alagamentos, quedas de árvore etc.) e não reportam nova enchente neste dia 22, indicando que foi o mesmo incidente climático. |
| **2019-03-10 e 2019-03-11**          | 1                            | 10/03/2019 e 11/03/2019              | Apenas um evento contínuo: a tempestade que começou na noite de 10 de março e se estendeu até a madrugada/manhã de 11 de março de 2019. Esse episódio causou graves enchentes e vítimas, incluindo **dois óbitos em Santo André** ([Chega a 12 o número de mortos por causa de forte chuva em SP | Agência Brasil](https://agenciabrasil.ebc.com.br/geral/noticia/2019-03/chega-12-o-numero-de-mortos-por-causa-de-forte-chuva-em-sp#:~:text=O%20Corpo%20de%20Bombeiros%20atualizou,Image%20%20101)). O dia 11/03 não teve outra chuva intensa além da já ocorrida entre dia 10 e 11; as notícias focam no balanço dessa mesma tragédia. |
| **2022-01-03 e 2022-01-04**          | 1                            | 03/01/2022                           | Houve um só incidente de alagamento, em 3 de janeiro de 2022 (primeiro dia útil do ano). Uma tempestade de cerca de uma hora na tarde desse dia (por volta de 13h30) despejou ~60 mm de chuva em Santo André ([Chuva de uma hora no 1º dia útil de 2022 mais uma vez castiga o Grande ABC - 03/01/2022 | Diário do Grande ABC](https://www.dgabc.com.br/Noticia/3824045/chuva-de-uma-hora-no-1-dia-util-de-2022-mais-uma-vez-castiga-o-grande-abc#:~:text=A%20tempestade%20que%20mais%20uma,da%20cidade%20informou%2036%20mm)), gerando diversos pontos de alagamento. No dia 04/01 não ocorreram novas enchentes – as informações divulgadas ainda se referiam aos alagamentos do dia 3. |
| **2022-03-06, 2022-03-07 e 2022-03-08** | 1                         | 06/03/2022                           | Um único evento confirmado, no dia 6 de março de 2022. Nesse domingo, uma forte chuva atingiu Santo André (e região), resultando em alagamentos. Os dias 07 e 08/03 não trouxeram novos temporais severos na cidade; as notícias nesses dias continuaram tratando dos impactos e da resposta ao evento de 06/03. |
| **2023-01-15 e 2023-01-16**          | 1                            | 15/01/2023                           | Apenas um evento de enchente ocorreu em 15 de janeiro de 2023. Na tarde de domingo (15) Santo André sofreu com chuva extrema – houve alagamentos e até uma fatalidade (um morador morreu após queda de árvore devido ao temporal) ([Santo André: Queda de árvore após chuva mata homem, no ABC paulista](https://noticias.uol.com.br/ultimas-noticias/agencia-estado/2023/01/16/queda-de-arvore-apos-chuva-mata-homem-em-santo-andre-no-abc-paulista.htm#:~:text=Um%20homem%20de%2054%20anos,vizinhos%20ao%20Corpo%20de%20Bombeiros)). No dia 16/01, embora tenha chovido novamente de forma moderada, não se registrou um novo evento crítico de alagamento; as atenções estavam voltadas às consequências do desastre do dia 15. |

Conclusão -> df_alagamentos_sem_subsequentes é o correto a se usar.

# Salva dados de alagamentos

* Salva datas em arquivo csv

In [ ]:
df_alagamentos_sem_subsequentes.to_csv("alagamentos_confirmados.csv", index=False)

# Cria csv chamados por bacia

* Verifica quantos chamados verificados nao existem no df de chamados
* Cria coluna one hot para cada bacia
* Junta com chamados base

In [ ]:
import unicodedata

def strip_accents(s):
    """
    Remove accent marks and diacritics from a string using Unicode normalization.

    This function decomposes the input string using NFD Unicode normalization and
    removes all combining character markers (accent/diacritic marks) to return
    a simplified ASCII-friendly version of the text.

    Parameters:
    ----------
    s : str
        The input string containing potential accented characters or diacritics.

    Returns:
    -------
    str
        The processed string with accent marks removed while preserving original characters.
    """
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                  if unicodedata.category(c) != 'Mn')


def rename_neighborhood_list(name_list):
    """
    Standardize neighborhood names by converting to uppercase, replacing common terms,
    and removing accents.

    This function processes a list of neighborhood names by:
    1. Converting all characters to uppercase
    2. Replacing common prefixes with abbreviations (VILA → VL, JARDIM → JD, etc.)
    3. Stripping accent marks using the strip_accents utility
    4. Modifies AND returns the input list object (operates in-place)

    Parameters:
    ----------
    name_list : list[str]
        A list of neighborhood name strings to be standardized in-place.

    Returns:
    -------
    list[str]
        The modified input list containing standardized neighborhood names. Common
        replacements include:
        - "VILA" becomes "VL"
        - "JARDIM" becomes "JD"
        - "PARQUE" becomes "PQ"

    Note:
    -----
    This function modifies the original list object while returning it. Consider making
    a copy if the original list needs to be preserved.
    """
    for i, name in enumerate(name_list):
        name_list[i] = strip_accents(name.upper().replace("VILA", "VL").replace("JARDIM", "JD").replace("PARQUE", "PQ"))
    return name_list

In [ ]:
# prompt: create a groupby agregator that uses a lambda to sum the strings with "; " in between. Apply that to the groupby in the cell above

groupby_aggregator = lambda x: ";".join(x)
df_chamados_daily = df_chamados_renamed.groupby("dt").agg(groupby_aggregator).reset_index()
df_chamados_daily['dt'] = pd.to_datetime(df_chamados_daily.dt, dayfirst=True)

In [ ]:
df_chamados_daily

,dt,end
0,2002-02-22,"RUA KALILI, 110 - VL FLORESTA"
1,2004-09-15,"AVN CAP MARIO TOLEDO DE CAMARGO, 6702 - JD IPA..."
2,2004-10-25,"RUA DR OSCAR BERNARDES, 79 - VL PALMARES;AVN D..."
3,2004-12-06,"AVN ACLIMACAO - ASS. HARADA, 90 - JD DO ESTADI..."
4,2005-01-05,"RUA AMORAS, DAS, 39 - VL PALMARES;RUA ROSARIA,..."
...,...,...
575,2023-03-14,"RUA CANDIDO RODRIGUES, 255 - VL JUNQUEIRA;RUA ..."
576,2023-03-15,"RUA ROCHA PITA, 193 - JD GUARARA"
577,2023-03-29,"RUA RANCHARIA, 8 - VL METALURGICA"
578,2023-04-27,"RUA IPIRANGA, 280 - PQ JOAO RAMALHO"


In [ ]:
# passar pelo strip_accents e depois rename_neighborhood_list
df_chamados_daily["endereco_tratado"] = df_chamados_daily["end"].apply(strip_accents).apply(lambda x: rename_neighborhood_list([x])[0])

In [ ]:
import re

def extract_bairros(enderecos: str) -> str:
    """
    Extrai o bairro de um endereço no formato "RUA ..., NÚMERO - BAIRRO".
    Se não houver hífen ou o bairro estiver ausente, retorna "SEM BAIRRO".
    """
    # O padrão procura um hífen seguido de espaço(s) e captura todo o restante da string.
    enderecos = enderecos.split(";")
    if not enderecos:
      return "SEM BAIRRO"
    bairros = []
    for end in enderecos:
      padrao = r'-\s*(.+)$'
      resultado = re.search(padrao, end)
      if resultado:
          bairro = resultado.group(1).strip()
          bairros.append(bairro)
    return ";".join(bairros)


In [ ]:
df_chamados_daily["bairro"] = df_chamados_daily["endereco_tratado"].apply(extract_bairros)
df_chamados_daily

,dt,end,endereco_tratado,bairro
0,2002-02-22,"RUA KALILI, 110 - VL FLORESTA","RUA KALILI, 110 - VL FLORESTA",VL FLORESTA
1,2004-09-15,"AVN CAP MARIO TOLEDO DE CAMARGO, 6702 - JD IPA...","AVN CAP MARIO TOLEDO DE CAMARGO, 6702 - JD IPA...",JD IPANEMA;VL VITORIA;VL HUMAITA;VL AMERICA;VL...
2,2004-10-25,"RUA DR OSCAR BERNARDES, 79 - VL PALMARES;AVN D...","RUA DR OSCAR BERNARDES, 79 - VL PALMARES;AVN D...",VL PALMARES;VL ASSUNCAO
3,2004-12-06,"AVN ACLIMACAO - ASS. HARADA, 90 - JD DO ESTADI...","AVN ACLIMACAO - ASS. HARADA, 90 - JD DO ESTADI...","ASS. HARADA, 90 - JD DO ESTADIO;ASS. HARADA, 9..."
4,2005-01-05,"RUA AMORAS, DAS, 39 - VL PALMARES;RUA ROSARIA,...","RUA AMORAS, DAS, 39 - VL PALMARES;RUA ROSARIA,...",VL PALMARES;VL SACADURA CABRAL;VL JOAO RAMALHO
...,...,...,...,...
575,2023-03-14,"RUA CANDIDO RODRIGUES, 255 - VL JUNQUEIRA;RUA ...","RUA CANDIDO RODRIGUES, 255 - VL JUNQUEIRA;RUA ...",VL JUNQUEIRA;VL METALURGICA
576,2023-03-15,"RUA ROCHA PITA, 193 - JD GUARARA","RUA ROCHA PITA, 193 - JD GUARARA",JD GUARARA
577,2023-03-29,"RUA RANCHARIA, 8 - VL METALURGICA","RUA RANCHARIA, 8 - VL METALURGICA",VL METALURGICA
578,2023-04-27,"RUA IPIRANGA, 280 - PQ JOAO RAMALHO","RUA IPIRANGA, 280 - PQ JOAO RAMALHO",PQ JOAO RAMALHO


In [ ]:
# Definindo as listas de bairros para cada bacia
bacia_meninos = [
    'VL PALMARES', 'VL SACADURA CABRAL', 'VL AQUILINO', 'VL PRINCIPE DE GALES',
    'VL GUIOMAR', 'VL VALPARAISO', 'VL FLORESTA', 'JD BOM PASTOR', 'VL SCARPELLI',
    'PINHEIRINHO', 'JD STELLA', 'JD JAMAICA', 'JD CRISTIANE', 'JD LAS VEGAS', 'JD ALVORADA'
]

bacia_oratorio = [
    'VL METALURGICA', 'VL CAMILOPOLIS', 'JD UTINGA', 'JD DAS MARAVILHAS',
    'VL LUCINDA', 'PQ ORATORIO', 'PQ ERASMO ASSUNCAO', 'PQ NOVO ORATORIO', 'JD RINA',
    'PQ CAPUAVA', 'JD SANTO ALBERTO', 'JD ITAPOAN', 'JD ANA MARIA', 'POLO PETROQUIMICO DE CAPUAVA'
]

bacia_tamanduatei = [
    'CAMPESTRE', 'SANTA MARIA', 'JD', 'VL ALPINA', 'VL GUIOMAR', 'VL ALICE',
    'VL BASTOS', 'JD BELA VISTA', 'CENTRO', 'CASA BRANCA', 'VL GILDA', 'PARAISO',
    'VL ASSUNCAO', 'VL ALZIRA', 'SILVEIRA', 'VL LINDA', 'VL HELENA', 'JD DO ESTADIO',
    'JD SANTA CRISTINA', 'JD GUARARA', 'SITIO DOS VIANAS', 'JD CIPRESTE', 'JD IRENE',
    'VL JOAO RAMALHO', 'JD VL RICA', 'CATA PRETA', 'JD SANTO ANDRE CDHU', 'JD SANTO ANDRE CDHU',
    'VL LUZITA', 'JD TELLES DE MENEZES', 'VL SUICA', 'CONDOMINIO MARACANA', 'VL LUTECIA',
    'VL TIBIRICA', 'VL VITORIA', 'JD IPANEMA', 'VL GUARACIABA', 'VL PROGRESSO', 'PQ GERASSI',
    'CIDADE SAO JORGE', 'CENTREVILLE', 'VL GUARANI', 'VL HUMAITA', 'PQ MARAJOARA',
    'VL HOMERO THON', 'VL AMERICA', 'VL PIRES', 'SILVEIRA', 'VL ALZIRA', 'NOVO HOMERO THON',
    'VARZEA DO TAMANDUATEI', 'JD ALZIRA FRANCO', 'PQ JACATUBA', 'VL CURUCA', 'BANGU',
    'PQ DAS NACOES', 'SANTA TEREZINHA', 'VL FRANCISCO MATARAZZO', 'JD SANTO ANTONIO',
    'VL CAMILOPOLIS', 'VL METALURGICA'
]

bacia_guarara = [
    'CATA PRETA', 'VL JOAO RAMALHO', 'JD VL RICA', 'JD SANTO ANDRE CDHU',
    'JD SANTO ANDRE', 'JD IRENE', 'SITIO DOS VIANAS', 'JD GUARARA', 'VL LUZITA',
    'VL SUICA', 'JD SANTA CRISTINA', 'JD TELLES DE MENEZES', 'VL LUTECIA',
    'JD DO ESTADIO', 'VL JUNQUEIRA', 'VL VITORIA', 'VL TIBIRICA', 'JD IPANEMA',
    'VL HELENA', 'VL PROGRESSO', 'VL PIRES', 'VL HUMAITA', 'SILVEIRA', 'VL AMERICA',
    'VL HOMERO THON', 'CASA BRANCA', 'VARZEA DO TAMANDUATEI', 'NOVO HOMERO THON'
]

def one_hot_bacia(bairros: str) -> tuple:
    """
    Recebe uma string de bairro e retorna uma tupla com 5 booleanos:
    (Bacia Tamanduatei Central, Bacia Guarara, Bacia Oratorio, Bacia Meninos, Nenhum)
    """
    tamanduatei, guarara, oratorio, meninos, nenhum = False, False, False, False, False
    if bairros == "SEM BAIRROS":
      nenhum = True
      return (tamanduatei, guarara, oratorio, meninos, nenhum)

    bairros = bairros.split(";")
    for bairro in bairros:
      bairro = bairro.strip().upper()
      if not tamanduatei:
        tamanduatei = bairro in bacia_tamanduatei
      if not guarara:
        guarara = bairro in bacia_guarara
      if not oratorio:
        oratorio   = bairro in bacia_oratorio
      if not meninos:
        meninos    = bairro in bacia_meninos

    return (tamanduatei, guarara, oratorio, meninos, nenhum)


In [ ]:
df_chamados_daily[['bacia_tamanduatei', 'bacia_guarara', 'bacia_oratorio', 'bacia_meninos', 'bacia_outras']] = df_chamados_daily["bairro"].apply(lambda b: pd.Series(one_hot_bacia(b)))

In [ ]:
df_chamados_final = df_chamados_daily.copy()

df_chamados_final.drop(columns=["end", "endereco_tratado", "bairro"])

,dt,bacia_tamanduatei,bacia_guarara,bacia_oratorio,bacia_meninos,bacia_outras
0,2002-02-22,False,False,False,True,False
1,2004-09-15,True,True,False,False,False
2,2004-10-25,True,False,False,True,False
3,2004-12-06,False,False,False,True,False
4,2005-01-05,True,True,False,True,False
...,...,...,...,...,...,...
575,2023-03-14,True,True,True,False,False
576,2023-03-15,True,True,False,False,False
577,2023-03-29,True,False,True,False,False
578,2023-04-27,False,False,False,False,False


In [ ]:
# prompt: create a new df: df_alagamentos_chamados. This will be a copy of df_alagamentos_sem_subsequentes, but with the [bacia_tamanduatei, bacia_guarara, bacia_oratorio, bacia_meninos and bacia_outras] columns from df_chamados_final. If the dt doesn't have an equivalent in df_chamados_final, check the day before and the day after and add the first hit

import pandas as pd
df_alagamentos_chamados = df_alagamentos_sem_subsequentes.copy()

# Merge with df_chamados_final based on 'dt'
df_alagamentos_chamados = df_alagamentos_chamados.merge(
    df_chamados_final[['dt', 'bacia_tamanduatei', 'bacia_guarara', 'bacia_oratorio', 'bacia_meninos', 'bacia_outras']],
    on='dt',
    how='left'
)

# Function to find the closest date with data in df_chamados_final
def find_closest_bacia_data(date, df_chamados, columns):
    # Check the current date
    current_day_data = df_chamados[df_chamados['dt'] == date][columns]
    if not current_day_data.empty:
        return current_day_data.iloc[0]

    # Check the day before
    prev_day = date - pd.Timedelta(days=1)
    prev_day_data = df_chamados[df_chamados['dt'] == prev_day][columns]
    if not prev_day_data.empty:
        return prev_day_data.iloc[0]

    # Check the day after
    next_day = date + pd.Timedelta(days=1)
    next_day_data = df_chamados[df_chamados['dt'] == next_day][columns]
    if not next_day_data.empty:
        return next_day_data.iloc[0]

    # If no data found within +/- 1 day, return default (False for all bacias)
    return pd.Series([False] * len(columns), index=columns)

# Fill missing bacia data by checking adjacent days
bacia_columns = ['bacia_tamanduatei', 'bacia_guarara', 'bacia_oratorio', 'bacia_meninos', 'bacia_outras']

for col in bacia_columns:
    missing_dates = df_alagamentos_chamados[df_alagamentos_chamados[col].isna()]['dt']
    for date in missing_dates:
        closest_data = find_closest_bacia_data(date, df_chamados_final, bacia_columns)
        df_alagamentos_chamados.loc[df_alagamentos_chamados['dt'] == date, bacia_columns] = closest_data.values

# Convert boolean columns to integer (0 or 1) if needed
for col in bacia_columns:
    df_alagamentos_chamados[col] = df_alagamentos_chamados[col].astype(bool).astype(int)

In [ ]:
df_alagamentos_chamados

,dt,ano,mes,dia,date_diff,bacia_tamanduatei,bacia_guarara,bacia_oratorio,bacia_meninos,bacia_outras
0,2016-01-10,2016,1,10,NaN,1,1,0,0,0
1,2016-01-15,2016,1,15,4.0,0,0,0,0,0
2,2016-02-05,2016,2,5,20.0,1,0,1,0,0
3,2016-02-15,2016,2,15,10.0,1,1,0,0,0
4,2016-02-24,2016,2,24,8.0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...
74,2024-02-02,2024,2,2,14.0,0,0,0,0,0
75,2024-02-24,2024,2,24,22.0,0,0,0,0,0
76,2024-12-27,2024,12,27,307.0,0,0,0,0,0
77,2025-01-31,2025,1,31,35.0,0,0,0,0,0


In [ ]:
df_alagamentos_chamados["total"] = df_alagamentos_chamados[["bacia_tamanduatei", "bacia_guarara", "bacia_oratorio", "bacia_meninos", "bacia_outras"]].sum(axis=1)

In [ ]:
df_alagamentos_chamados.total.value_counts()

,count
total,
2,27
1,20
0,18
3,12
4,2


In [ ]:
df_alagamentos_chamados.to_csv("alagamentos_bacias.csv", index=False)